In [2]:
import pandas as pd 
from sklearn.feature_extraction.text import TfidfVectorizer

In [3]:
dataset=pd.read_csv("../data/processed/common/common_publications_final.csv")
print(dataset.shape)
print(dataset.columns)

/tmp/ipykernel_294029/3178825721.py:1: DtypeWarning: Columns (0: source_institution_id, 1: source_datestamp, 2: openalex_id, 3: pdf_url, 4: keywords, 5: author_affiliations, 6: author_orcids, 7: sri_lankan_authors, 8: contributors, 9: institutions, 10: sri_lankan_institutions, 11: countries, 12: journal, 13: source_type, 14: issn, 15: issn_l, 16: volume, 17: issue, 18: first_page, 19: last_page, 20: article_number, 21: license, 22: license_url, 23: is_oa, 24: concepts, 25: topics, 26: primary_topic, 27: primary_field, 28: primary_subfield, 29: primary_domain, 30: funder_name, 31: funder_doi, 32: funder_identifier, 33: funder_award, 34: source_set_specs, 35: raw_identifiers, 36: citation_count_divergence_flag, 37: reference_count_divergence_flag, 38: keywords_search_text) have mixed types. Specify dtype option on import or set low_memory=False.
  dataset=pd.read_csv("../data/processed/common/common_publications_final.csv")


(147859, 71)
Index(['source_dataset', 'source_institution_id', 'source_record_id',
       'source_datestamp', 'openalex_id', 'doi', 'url', 'pdf_url', 'title',
       'abstract', 'keywords', 'publication_year', 'publication_date', 'type',
       'authors', 'author_count', 'author_affiliations', 'author_orcids',
       'sri_lankan_authors', 'contributors', 'institutions',
       'sri_lankan_institutions', 'countries', 'publisher', 'journal',
       'source_type', 'issn', 'issn_l', 'volume', 'issue', 'first_page',
       'last_page', 'article_number', 'language', 'license', 'license_url',
       'oa_status', 'is_oa', 'citation_count', 'reference_count', 'concepts',
       'topics', 'primary_topic', 'primary_field', 'primary_subfield',
       'primary_domain', 'funder_name', 'funder_doi', 'funder_identifier',
       'funder_award', 'source_set_specs', 'raw_identifiers',
       'citation_count_difference_oa_minus_crossref',
       'citation_count_divergence_flag',
       'reference_count_di

In [4]:
pred_df=dataset[dataset["primary_field"].isna()]
print(f"pred: {pred_df.shape[0]}")
df=dataset[dataset["primary_field"].notna()]
print(f"df: {df.shape[0]}")

pred: 75235
df: 72624


In [5]:
print(df.shape[0]+pred_df.shape[0]==dataset.shape[0])

True


In [6]:
def build_lookup(taxonomy):
    lookup={}
    for domain,fields in taxonomy.items():
        for field,subfields in fields.items():
            for subfield in subfields:
                lookup[(domain,subfield)]=field
    return lookup


In [7]:
import json
with open("../category_hierarchy.json","r",encoding="utf-8") as f:
    taxonomy = json.load(f)
lookup = build_lookup(taxonomy)

In [8]:
text_cols=["title","abstract","topics","keywords","concepts"]

In [9]:
def processing(df):
    df["text"] = (
        df[text_cols]
        .fillna("")
        .astype(str)
        .agg(" ".join, axis=1)
        .str.replace(r"\s+", " ", regex=True)
        .str.strip()
    )
    df["domain"]=df["primary_field"]
    df["subfield"]=df["primary_subfield"]
    
    df["field"] = df.apply(lambda r:lookup.get((r["domain"],r["subfield"])),axis=1)

In [10]:
from sklearn.model_selection import train_test_split

In [11]:
# primary fields - 26 ----------------> domain
# primary subfields - 254     --------> subfield
# final domain -> field -> subfield
# field ------------------------------> (domain,subfield)


In [11]:
def data_split(col):
    X_train, X_test, y_train, y_test = train_test_split(
        df["text"], df[col], test_size=0.15, random_state=42, stratify=df[col]
    )
    return X_train,X_test,y_train,y_test

In [12]:
processing(df)

In [13]:
#domin
X_train, X_test, y_train, y_test=data_split("domain")

In [14]:
vectorizer = TfidfVectorizer(
    lowercase=True,
    stop_words="english",
    ngram_range=(1,2),
    min_df=2,
    max_df=0.95,
    sublinear_tf=True,
    max_features=50000,
    
)

In [16]:
# X_train_tfidf=vectorizer.fit_transform(X_train)
# X_test_tfidf=vectorizer.transform(X_test)

In [15]:
from sklearn.svm import LinearSVC
from sklearn.pipeline import Pipeline



In [16]:
base_svm = LinearSVC(
    class_weight="balanced",
    max_iter=5000,
    random_state=42,
    dual="auto"
    )

In [17]:
pipe= Pipeline([
    ("tfidf",vectorizer),
    ("svm",base_svm)
])

In [18]:
from sklearn.model_selection import GridSearchCV

param_grid = {"svm__C": [0.1, 1, 10]}
grid = GridSearchCV(pipe, param_grid, cv=3, scoring="f1_macro", n_jobs=-1)


In [24]:
grid.fit(X_train, y_train)


,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",Pipeline(step...m_state=42))])
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'svm__C': [0.1, 1, ...]}"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion <scoring_api_overview>` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",'f1_macro'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- an iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide <cross_validation>` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",3
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`sphx_glr_auto_examples_model_selection_plot_grid_search_digits.py`to see how to design a custom selection strategy using a callablevia `refit`.See :ref:`this example<sphx_glr_auto_examples_model_selection_plot_grid_search_refit_callable.py>`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",True
,"verbose verbose: int, default=0Controls the verbosity of information printed during fitting, with 

In [30]:
best_model = grid.best_estimator_
print(grid.best_params_)

{'svm__C': 1}


In [31]:
print(best_model)

Pipeline(steps=[('tfidf',
                 TfidfVectorizer(max_df=0.95, max_features=50000, min_df=2,
                                 ngram_range=(1, 2), stop_words='english',
                                 sublinear_tf=True)),
                ('svm',
                 LinearSVC(C=1, class_weight='balanced', max_iter=5000,
                           random_state=42))])


In [32]:
import pickle
pickle.dump(best_model, open("domain_pipeline.pkl", "wb"))

In [33]:
grid.cv_results_

{'mean_fit_time': array([40.62971505, 47.63299441, 66.30197461]),
 'std_fit_time': array([1.78272219, 1.01452937, 0.59511748]),
 'mean_score_time': array([4.9048605 , 3.35969893, 2.50207663]),
 'std_score_time': array([0.63143669, 0.05859802, 0.04416281]),
 'param_svm__C': masked_array(data=[0.1, 1.0, 10.0],
              mask=[False, False, False],
        fill_value=1e+20),
 'params': [{'svm__C': 0.1}, {'svm__C': 1}, {'svm__C': 10}],
 'split0_test_score': array([0.67613338, 0.71715445, 0.70120251]),
 'split1_test_score': array([0.67283633, 0.72152142, 0.70903798]),
 'split2_test_score': array([0.66878153, 0.71778149, 0.70224859]),
 'mean_test_score': array([0.67258375, 0.71881912, 0.70416303]),
 'std_test_score': array([0.00300669, 0.00192788, 0.00347347]),
 'rank_test_score': array([3, 1, 2], dtype=int32)}

In [35]:
print(f"best score: {grid.best_score_}")



best score: 0.7188191193830464


In [36]:
preds= best_model.predict(X_test)

In [38]:
from sklearn.metrics import classification_report,accuracy_score

print(accuracy_score(y_test,preds))


0.8050302919038003


In [40]:
report_dict = classification_report(y_test, preds, output_dict=True, zero_division=0)
report_df = pd.DataFrame(report_dict).transpose()
report_df

,precision,recall,f1-score,support
Agricultural and Biological Sciences,0.815402,0.811725,0.813559,887.00000
Arts and Humanities,0.719251,0.825153,0.768571,326.00000
"Biochemistry, Genetics and Molecular Biology",0.747012,0.718391,0.732422,522.00000
"Business, Management and Accounting",0.799694,0.847650,0.822974,617.00000
Chemical Engineering,0.600000,0.333333,0.428571,9.00000
Chemistry,0.606299,0.706422,0.652542,109.00000
Computer Science,0.843511,0.843511,0.843511,786.00000
Decision Sciences,0.659341,0.727273,0.691643,165.00000
Dentistry,0.805970,0.915254,0.857143,59.00000
Earth and Planetary Sciences,0.729730,0.771429,0.750000,245.00000


In [41]:
report = classification_report(y_test, preds, zero_division=0)


with open("classification_report.txt", "w") as f:
    f.write(report)

In [32]:
def process_domain(domain):
    sub_df=df[df["domain"]==domain]
    n_classes=sub_df["subfield"].nunique()
    class_counts=sub_df["subfield"].value_counts()
    print(f"classes: {n_classes}")
    print(class_counts)
    return sub_df

In [ ]:
domains = df["domain"].unique()
for domain in domains:
    sub_df=process_domain(domain)
    if len(sub_df) < 20:
        pass
    break


classes: 42
subfield
Public Health, Environmental and Occupational Health    2270
Surgery                                                 1240
Epidemiology                                            1095
Infectious Diseases                                      779
Endocrinology, Diabetes and Metabolism                   763
Pulmonary and Respiratory Medicine                       735
Physiology                                               600
Pediatrics, Perinatology and Child Health                571
Radiology, Nuclear Medicine and Imaging                  516
Oncology                                                 509
Cardiology and Cardiovascular Medicine                   498
Complementary and alternative medicine                   388
Pharmacology                                             355
Pathology and Forensic Medicine                          352
Rheumatology                                             347
Psychiatry and Mental health                             344
Obs